In [25]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
import pickle

In [26]:
data_dir = "../data"
x_list = []
y_list = []
batch_size = 32

In [27]:
for file in os.listdir(data_dir):
    path = os.path.join(data_dir, file)
    with open(path, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')

        x_list.append(data_dict[b'data'])
        y_list.extend(data_dict[b'labels'])

C:\Users\xavif\AppData\Local\Temp\ipykernel_20972\250140131.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data_dict = pickle.load(fo, encoding='bytes')


In [28]:
X = np.concatenate(x_list)
Y = np.array(y_list)

In [29]:
X = X.reshape(-1, 3,32,32)
X = X.transpose(0,2,3,1)
X = X.astype(float) / 255.0

In [30]:
y = to_categorical(Y, 10)

In [31]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))

dataset = dataset.shuffle(buffer_size=len(X), seed=42)

In [32]:
train_size = int(0.8*len(X))

In [33]:
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)

In [45]:
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [35]:
IMG_SIZE = (32,32) + (3,)

In [36]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')

C:\Users\xavif\AppData\Local\Temp\ipykernel_20972\846687965.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')


In [ ]:
base_model.summary()

# Congelación de capas

In [38]:
base_model.trainable = False

In [39]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x= layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

In [41]:
model = models.Model(inputs=base_model.input, outputs=output)

In [42]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset, 
    validation_data=val_dataset,
    epochs=10, 

)

Epoch 1/10


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.